# clue_sft

> Phase 12, the warm start: the rule agent's moves (sharing every clue nobody has shared, so each move depends only on what the prompt shows) as chat examples with the note first (for a submit, the candidates its known clues leave), LoRA fine-tuning on them, merging into the warm-start model, and the warm-start check (solve rate with every clue held, invalid replies). Training needs the `[train]` extra; the offline tests do not.

In [ ]:
#| default_exp clue_sft

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np

from explRL.clue_agents import known
from explRL.clue_grid import BOARD, CLUE_SETTINGS
from explRL.clue_policy import renderPrompt, userText
from explRL.clue_roll import coopRows, rollout
from explRL.clue_train import ClueEpisodes, teacherAction
from explRL.llm_runs import LLMConfig
from explRL.local_policy import dtypeFor, pickDevice
from explRL.multi import spawnRngs
from explRL.rl_torch import replyLogps, saveMemory


def _clue(p, r):
    return f'x mod {p} = {r}'


def teacherNote(board, i, channel, reply):
    "The teacher's note: for a submit, the candidates its known clues leave (the rules' method); else a short reason."
    a = reply['action']
    if a == 'submit':
        k = board.opened[i]
        task = board.tasks[k].task
        kn = sorted(((task.primes[j], r) for j, r in known(i, board, channel)[k].items()), reverse=True)
        (p, r), xs = kn[0], list(range(kn[0][1], board.R, kn[0][0]))
        s = f'{_clue(p, r)} below {board.R}: ' + ', '.join(map(str, xs))
        for q, u in kn[1:]:
            xs = [y for y in xs if y % q == u]
            s += f'; of these {_clue(q, u)}: ' + ', '.join(map(str, xs))
        return s
    if a == 'open':
        k = int(reply['target'][1:])
        return f"open {reply['target']}: {len(known(i, board, channel)[k])} of {board.m} clues known, value {board.tasks[k].value:g}"
    return f"share my clue for {reply['target']}" if a == 'share' else 'nothing to do'


def teacherReply(board, i, channel, rng):
    "The teacher's move (share 'always') as reply text: its note first, then action, target, answer and clue."
    a = teacherAction(board, i, channel, rng, share='always')
    return json.dumps(dict(notes=teacherNote(board, i, channel, a), **{k: a[k] for k in ('action', 'target', 'answer', 'clue')}))


def sftExamples(seeds, sizes=(1, 2, 4), turns=20, settings=CLUE_SETTINGS, boards=False):
    """Warm-start chat examples (system, user, assistant, key): the teacher in every slot, blackboard boards (no channel
    at N = 1), prompts with notes memory as rollout renders them. boards=True also returns the played boards."""
    out, played = [], []
    for N in sizes:
        ep = ClueEpisodes(len(seeds), N, {k: settings[k] for k in BOARD}, turns=turns,
                          channel='blackboard' if N > 1 else 'none', seeds=seeds)
        rngs, notes, nxt = [spawnRngs(s + 4000, N) for s in seeds], [[''] * N for _ in seeds], ep.reset()
        while any(x is not None for x in nxt):
            replies = [None] * len(seeds)
            for b, x in enumerate(nxt):
                if x is not None:
                    i, obs = x
                    text = teacherReply(ep.boards[b], i, ep.channels[b], rngs[b][i])
                    system, blocks = renderPrompt(ep.envs[b], obs, 'notes', notes[b][i])
                    out.append(dict(system=system, user=userText(blocks), assistant=text,
                                    key=f'{N}|{seeds[b]}|{i}|{ep.boards[b].tWorld:g}'))
                    replies[b] = json.loads(text)
                    notes[b][i] = replies[b]['notes']
            nxt, _, _ = ep.step(replies)
        played += ep.boards
    return (out, played) if boards else out


@dataclass
class SFTConfig:
    model: str
    out: str
    loraR: int = 16
    loraAlpha: int = 32
    dropout: float = 0.05
    lr: float = 1e-4
    epochs: int = 1
    batch: int = 16
    microBatch: int = 4
    maxLen: int = 1536
    maxSteps: int = 0           # 0: one pass over the examples per epoch
    device: str = 'auto'
    seed: int = 0
    gradCkpt: bool = True       # gradient checkpointing (less memory, about a third more compute)


def libVersions():
    "Versions of the libraries a training run used (recorded with its configuration)."
    from importlib.metadata import PackageNotFoundError, version
    out = {}
    for p in ('torch', 'transformers', 'peft', 'vllm'):
        try:
            out[p] = version(p)
        except PackageNotFoundError:
            out[p] = None
    return out


def loraConfig(r, alpha, dropout=0.0):
    from peft import LoraConfig
    return LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules='all-linear', task_type='CAUSAL_LM')


def loadModel(model, device, dtype=None):
    "(tokenizer, model on device); pad token set."
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(model)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok, AutoModelForCausalLM.from_pretrained(model, dtype=dtype or dtypeFor(device)).to(device)


def trainSFT(cfg, examples):
    "Fine-tune a LoRA adapter on the examples (loss on the replies only); -> the adapter's path. Logs metrics.jsonl."
    import torch
    from peft import get_peft_model
    torch.manual_seed(cfg.seed)
    dev, out = pickDevice(cfg.device), Path(cfg.out)
    out.mkdir(parents=True, exist_ok=True)
    tok, base = loadModel(cfg.model, dev)
    model = saveMemory(get_peft_model(base, loraConfig(cfg.loraR, cfg.loraAlpha, cfg.dropout)), cfg.gradCkpt).train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg.lr)
    order = np.random.default_rng(cfg.seed).permutation(np.tile(np.arange(len(examples)), cfg.epochs))
    steps = len(order) // cfg.batch if not cfg.maxSteps else min(cfg.maxSteps, len(order) // cfg.batch)
    with open(out / 'metrics.jsonl', 'w') as log:
        for s in range(steps):
            t0, idx, total = time.time(), order[s * cfg.batch:(s + 1) * cfg.batch], 0.0
            for a in range(0, len(idx), cfg.microBatch):
                chunk = [dict(system=e['system'], blocks=[dict(type='text', text=e['user'])], reply=e['assistant'], eos=True)
                         for e in (examples[n] for n in idx[a:a + cfg.microBatch])]
                lps = replyLogps(model, tok, chunk, cfg.maxLen, dev)     # logits of the reply positions only
                loss = -sum(lp.sum() for lp in lps) / sum(len(lp) for lp in lps) * len(chunk) / len(idx)
                loss.backward()
                total += loss.item()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            opt.zero_grad()
            log.write(json.dumps(dict(step=s + 1, loss=total, seconds=round(time.time() - t0, 3))) + '\n')
            log.flush()
    model.save_pretrained(out / 'adapter')
    (out / 'config.json').write_text(json.dumps(dict(asdict(cfg), versions=libVersions()), indent=1) + '\n')
    return str(out / 'adapter')


def mergeAdapter(model, adapter, out):
    "The warm-start model: the adapter merged into the base weights (bf16, on cpu), saved with the tokenizer to out."
    import torch
    from peft import PeftModel
    tok, base = loadModel(model, 'cpu', torch.bfloat16)
    PeftModel.from_pretrained(base, adapter).merge_and_unload().save_pretrained(out)
    tok.save_pretrained(out)
    return str(out)


def warmCheck(policy, seeds=range(1200, 1220), turns=30, settings=CLUE_SETTINGS):
    "N = 1 boards (every clue held): solve rate (solved / opened) and invalid-reply rate; passes at >= 0.8 and < 0.05."
    cfg = LLMConfig(task='clue', channel='none', N=1, turns=turns, world={k: settings[k] for k in BOARD})
    rows = coopRows(cfg, rollout(cfg, list(seeds), policy))
    opens, solved = sum(r['opens'] for r in rows), sum(r['solved'] for r in rows)
    calls, invalid = sum(r['calls'] for r in rows), sum(r['invalid'] for r in rows)
    rate, inv = solved / opens if opens else 0.0, invalid / calls if calls else 1.0
    return dict(solveRate=rate, invalid=inv, opens=opens, solved=solved, passed=rate >= 0.8 and inv < 0.05)

## Tests

In [ ]:
import json, re
from explRL.clue_policy import parseReply, scriptedPolicy
from explRL.clue_roll import coopRows, rollout
from explRL.llm_runs import LLMConfig

W = dict(m=2, r=1, R=30, value=(1, 2, 3), life=30, penalty=0.5)
ex, boards = sftExamples([20000, 20001], sizes=(1, 2), turns=12, boards=True)
assert len(ex) == 2 * 12 * (1 + 2) and len(boards) == 4
# AC-4: every reply parses, its note comes first, and submit notes list exactly the candidates
acts = set()
for e in ex:
    r = parseReply(e['assistant'])
    assert list(r) == ['notes', 'action', 'target', 'answer', 'clue'] and r['action'] in ('open', 'submit', 'share', 'wait')
    acts.add(r['action'])
    if r['action'] == 'submit':
        parts = re.findall(r'x mod (\d+) = (\d+)[^:]*: ([\d, ]+)', r['notes'])
        (p, a, first), last = parts[0], parts[-1][2]
        assert [int(x) for x in first.split(', ')] == list(range(int(a), 30, int(p)))
        assert [int(x) for x in last.split(', ')] == [r['answer']]
assert {'open', 'submit', 'share'} <= acts
# replaying the examples' replies through rollout (notes memory) reproduces the teacher's boards
lookup = {(e['system'], e['user']): e['assistant'] for e in ex}
replay = scriptedPolicy(lambda s, b: lookup[(s, '\n\n'.join(x['text'] for x in b))])
for N, bs in ((1, boards[:2]), (2, boards[2:])):
    cfg = LLMConfig(task='clue', channel='blackboard' if N > 1 else 'none', N=N, turns=12, world=W)
    rows = coopRows(cfg, rollout(cfg, [20000, 20001], replay))
    assert [r['solved'] for r in rows] == [b.counts['solved'] for b in bs] and all(r['invalid'] == 0 for r in rows)
    if N == 1:
        assert all(b.counts['solved'] >= b.counts['opens'] - 1 and b.counts['wrong'] == 0 for b in bs)
# the warm-start check: a solver that reads only the text passes, a policy that only waits fails
def solver(system, blocks):
    o = blocks[-1]['text']
    m = re.search(r'Your open subtask: T(\d+), open\. Find x with 0 <= x < (\d+)\. Clues you know: (.*)\.', o)
    if not m:
        return json.dumps(dict(action='open', target=re.search(r'\): (T\d+)', o).group(1), answer=0, clue='', notes=''))
    cl = [tuple(map(int, c)) for c in re.findall(r'x mod (\d+) = (\d+)', m.group(3))]
    xs = [y for y in range(int(m.group(2))) if all(y % p == r for p, r in cl)]
    return json.dumps(dict(action='submit', answer=xs[0], target='', clue='', notes='') if len(xs) == 1 else dict(action='wait'))
ok = warmCheck(scriptedPolicy(solver), seeds=range(1200, 1203), turns=20)
assert ok['passed'] and ok['solveRate'] >= 0.8 and ok['invalid'] == 0, ok
bad = warmCheck(scriptedPolicy(lambda s, b: '{"action": "wait"}'), seeds=range(1200, 1202), turns=10)
assert not bad['passed'] and bad['opens'] == 0
assert set(libVersions()) == {'torch', 'transformers', 'peft', 'vllm'}
assert SFTConfig('m', 'o').batch == 16 and asdict(SFTConfig('m', 'o'))['loraR'] == 16

In [ ]:
#| train
# With the [train] extra, no download: the warm start, merge and check on the tiny random model (the loss falls)
import tempfile
from explRL.local_policy import LocalPolicy
from explRL.tiny_model import tinyModel
with tempfile.TemporaryDirectory() as d:
    tiny = tinyModel(d + '/tiny')
    a = trainSFT(SFTConfig(tiny, d + '/sft', maxSteps=6, batch=8, microBatch=4, lr=1e-3), sftExamples(range(20000, 20003), sizes=(1, 2), turns=10))
    losses = [json.loads(x)['loss'] for x in open(d + '/sft/metrics.jsonl')]
    assert len(losses) == 6 and losses[-1] < losses[0], losses
    res = warmCheck(LocalPolicy(mergeAdapter(tiny, a, d + '/merged'), maxNew=24), seeds=range(1200, 1202), turns=4)
    assert set(res) == {'solveRate', 'invalid', 'opens', 'solved', 'passed'}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()